# Parte 2 — Sensibilidad local (metodo de clase)

Saint-Venant 1D — Coeficientes de sensibilidad escalados (SSC) y validacion de las **5 suposiciones** del modelo de errores.

Basado en Michaelis-Menten adaptado al hidrograma en $x=L$.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t

from src.synthetic_data import generate_synthetic_data
from src.sensibilidad import (
    SSC, jacobian, make_model_callable,
    fit_parameters, ols_diagnostics, check_error_assumptions,
    bootstrap_parameters, PARAM_NAMES, ALPHA, DH,
)

## Datos sinteticos y modelo

In [ ]:
dataset = generate_synthetic_data(seed=42)
t = dataset.t
Yobs = dataset.q_obs
q_true = np.array(dataset.params_true)
fun = make_model_callable(t)
xobs = t
xpred = t  # misma malla temporal

print("Parametros verdaderos:", dict(zip(PARAM_NAMES, q_true)))

## Minimos cuadrados no lineales

In [ ]:
q0 = q_true * np.array([1.05, 0.95, 1.02, 0.98, 1.01])
q_ols, R, X, sigma = fit_parameters(Yobs, t, q0=q0)
se, se_rel, Cov, corr, ci_params = ols_diagnostics(q_ols, R, X, Yobs)
Ypred = fun(xobs, q_ols)

print(f"q estimado: {q_ols}")
print(f"Errores estandar relativos: {se_rel}")

## Numero de condicion y correlacion

In [ ]:
condX = np.linalg.cond(X)
print(f"Numero de condicion: {condX:.2e} (< 1e6 recomendado)")
print("Matriz de correlacion:")
print(np.round(corr, 3))

## Coeficientes de sensibilidad escalados (SSC)

In [ ]:
Xp = SSC(q_ols, xpred, fun, dh=DH)
X2 = jacobian(q_ols, xobs, fun, dh=DH)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(xpred / 3600, fun(xpred, q_ols), "k-", lw=2, label="Q simulado")
colors = ["c", "r", "g", "m", "orange"]
for i, name in enumerate(PARAM_NAMES):
    ax.plot(xpred / 3600, Xp[:, i], "--", color=colors[i], label=f"X'_{name}")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Tiempo (h)")
ax.set_ylabel("SSC (m3/s)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Verificacion de las 5 suposiciones

1. Errores aditivos  
2. Media del error = 0  
3. Varianza constante  
4. Errores no correlacionados  
5. Distribucion normal

In [ ]:
checks = check_error_assumptions(R, Ypred, t, X, sigma)
for msg in checks.messages:
    print(msg)

In [ ]:
# Sup.1: aditivos
fig, ax = plt.subplots()
ax.plot(Ypred, R, "bs", ms=5)
ax.axhline(0, color="r")
ax.set_xlabel("Prediccion Q (m3/s)")
ax.set_ylabel("Residuales")
ax.set_title("Sup.1 Errores aditivos")
plt.show()

In [ ]:
# Sup.2: media cero
print(f"Media de residuales: {np.mean(R):.3e}")

In [ ]:
# Sup.3: varianza constante
fig, ax = plt.subplots()
ax.plot(xobs / 3600, R, "bs", ms=5)
ax.axhline(0, color="r")
ax.set_xlabel("Tiempo (h)")
ax.set_ylabel("Residuales")
ax.set_title("Sup.3 Varianza constante")
plt.show()

In [ ]:
# Sup.4: no correlacion (criterio de cruces)
rescross = R[1:] * R[:-1]
count = np.sum(np.sign(rescross) < 0)
minrun = (len(R) + 1) / 2
if count >= minrun:
    print(f"OK: {count} cruces (minimo {minrun:.0f})")
else:
    print(f"REVISAR: {count} cruces (minimo {minrun:.0f})")

In [ ]:
# Sup.5: normalidad
fig, ax = plt.subplots()
ax.hist(R, density=True, bins=12, alpha=0.5, edgecolor="black")
xmin, xmax = ax.get_xlim()
xnorm = np.linspace(xmin, xmax, 100)
ax.plot(xnorm, norm.pdf(xnorm, 0, sigma), "k", lw=2)
ax.set_title("Sup.5 Normalidad")
plt.show()

## Bootstrap (opcional)

In [ ]:
q_boot, ci_b = bootstrap_parameters(Yobs, t, q_ols, Ypred, R, nboot=200, seed=42)
print("Intervalos bootstrap 95%:")
for i, name in enumerate(PARAM_NAMES):
    print(f"  {name}: [{ci_b[i,0]:.4f}, {ci_b[i,1]:.4f}]")

## Pipeline completo (script)

In [ ]:
from src.sensibilidad import run_sensitivity_analysis, save_results

result = run_sensitivity_analysis(dataset=dataset)
save_results(result, ROOT / "figures", ROOT / "data" / "synthetic")
print("Figuras guardadas en figures/")